In [ ]:
from pathlib import Path
import duckdb



In [ ]:
SQL_FILE = Path("analyze.sql")
data_fp = 'data/cps.parquet'
PARQUET = Path(data_fp)
try:
    con = duckdb.connect()
except Exception as e:
    print(f"Error connecting to DuckDB: {e}")
    raise




In [ ]:
all_col = """
'YEAR', 'SERIAL', 'MONTH', 'CPSID', 'ASECFLAG', 'ASECWTH', 'PERNUM', 'CPSIDP', 'CPSIDV', 'ASECWT', 'AGE', 'SEX', 'EMPSTAT', 'LABFORCE', 'OCC', 'IND', 'EDUC', 'INCWAGE'
"""
common_col = """
YEAR, SERIAL, CPSID, CPSIDP, AGE, SEX, EMPSTAT, LABFORCE, OCC, IND, EDUC, INCWAGE
"""
common_query = f"""
select {common_col} from 'data/cps.parquet'
"""

In [ ]:
# just seeing if we can read the parquet file
sql = f"""
select count(*) from '{data_fp}'
"""
print(con.sql(sql))

In [ ]:
# counting how many unique CPSIDP there are in the parquet file
sql = f"""
select count(distinct CPSIDP) from '{data_fp}'
"""
print(con.sql(sql))
# so since all of them are different, it suggests that none of the CPSIDP are the same accross the 2019 and 2024. thus, drop the column

common_col = """
YEAR, SERIAL, CPSID, AGE, SEX, EMPSTAT, LABFORCE, OCC, IND, EDUC, INCWAGE
"""

In [ ]:
# counting how many unique CPSI there are in the parquet file
sql = f"""
select count(distinct CPSID) from '{data_fp}'
"""
print(con.sql(sql))

# just looking through household data
sql = f"""
select CPSID, count(CPSID) as count_CPSID from '{data_fp}' group by CPSID having count(CPSID) > 1 order by count(CPSID) desc
"""
print(con.sql(sql))

# just looking through one specific cpsid for understanding
sql = f"""
select {common_col} from '{data_fp}' where CPSID = 20241308942800 order by YEAR
"""
print(con.sql(sql))
# this gotta be one big family including grand parents who are still working at the age of late 50s. this household have 3 generations -- interesting

### How has the unemployment rate for ages 22–25 changed by year?

In [ ]:
# LABFORCE = 2 means unemployed, 1 means employed, 3 means not in labor force.
# EMPSTAT = 10 means employed, 20 means unemployed, 30 means not in labor force.

# seeing all the data
sql = f"""
select {common_col} 
from '{data_fp}' 
where 
    AGE >= 22 and AGE <= 25 and 
    EMPSTAT >= 20 and EMPSTAT < 30
order by YEAR ;
"""
print(con.sql(sql))



In [ ]:
# seeing sepcific columns
sql = f"""
select year, count(*) as count, sum(ASECWT) as weighted_count
from '{data_fp}' 
where 
    AGE BETWEEN 22 AND 27 and 
    EMPSTAT IN (20,21,22)
group by year
order by YEAR;
"""
print(con.sql(sql))

In [ ]:
# now converting into pct
sql = f"""
SELECT YEAR,
  ROUND(100.0 * SUM(ASECWT) FILTER (WHERE EMPSTAT IN (20,21,22)) / NULLIF(SUM(ASECWT) FILTER (WHERE LABFORCE = 2), 0), 1) AS young_unemp_pct
FROM '{data_fp}'
WHERE AGE BETWEEN 22 AND 27
GROUP BY YEAR
ORDER BY YEAR;
"""
print(con.sql(sql))

### Do young workers (22–27) have higher unemployment than all workers (25–64)?

In [ ]:
# now converting into pct
sql = f"""
SELECT YEAR,
  ROUND(100.0 *
    SUM(ASECWT) FILTER (WHERE EMPSTAT IN (20,21,22))
    / NULLIF(SUM(ASECWT) FILTER (WHERE LABFORCE = 2), 0), 1) AS all_unemp_pct
FROM '{data_fp}'
WHERE AGE >= 22
GROUP BY YEAR
ORDER BY YEAR;
"""
print(con.sql(sql))